# 01. 유니버스 선정 (GICS 11 섹터 × 시가총액 Top 10)

**목적**: 2016-01 첫 영업일 기준 시가총액 상위 종목을 GICS 11 섹터별로 10개씩 선정하여
 **110개 종목으로 구성된 고정 유니버스**(시나리오 1)를 확정한다.

> **수집 기간**: `2010-01-01 ~ 2025-12-31`
 — 이 노트북은 유니버스 *선정* 기준 시점(2016-01-04)만 다루며,
 실제 가격 수집 범위는 `02_data_collection.ipynb`에서 설정한다.

**파이프라인**
1. Wikipedia S&P 500 현재 구성 종목 + GICS 섹터 로드
2. **SPDR SPY 보유 종목으로 교차검증** (신규)
3. 티커 리네임(META·PARA·FI 등) 정규화
4. 각 티커의 2016-01-04 시점 시가총액 추정
   - 1순위: `yfinance.Ticker.get_shares_full()` 가용 구간
   - 2순위: `current_shares ÷ Π(split_ratios since 2016-01-04)` 분할 역산
5. 섹터별 Top 10 선정, 경계 종목(10~12위) 로그
6. `data/universe.csv` 저장

**주의**
- 현재 S&P 500 리스트 기반이므로 2016년 당시 탈락한 종목은 누락됨 → 생존자 편향
- 시가총액 추정 루프는 yfinance API 500여 회 → 수 분~수십 분 소요,
 중간 캐시 저장으로 재실행 비용 최소화


---
## Section 0. 설정 및 함수 정의

In [1]:
# 표준 라이브러리 + 외부 3종(pandas, numpy, yfinance)만 사용
import os
import time
import pickle
import platform
from pathlib import Path
from typing import Optional

import io

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import requests

# 경로 설정 (노트북이 black_litterman/notebooks/ 에서 실행된다고 가정)
NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
DATA_DIR = PROJECT_DIR / 'data'
CACHE_DIR = DATA_DIR / 'cache'
DATA_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# 유니버스 기준 시점: 2016-01-01은 금요일 신정 휴장, 첫 영업일은 2016-01-04(월)
ANCHOR_DATE = pd.Timestamp('2016-01-04')

# GICS 11 섹터 표준 명칭 (Wikipedia 기준)
GICS_11_SECTORS = [
    'Energy', 'Materials', 'Industrials',
    'Consumer Discretionary', 'Consumer Staples',
    'Health Care', 'Financials',
    'Information Technology', 'Communication Services',
    'Utilities', 'Real Estate'
]


def setup_korean_font():
    """matplotlib 한글 폰트 설정 — OS별 분기."""
    if platform.system() == 'Windows':
        plt.rcParams['font.family'] = 'Malgun Gothic'
    elif platform.system() == 'Darwin':
        plt.rcParams['font.family'] = 'AppleGothic'
    else:
        try:
            import koreanize_matplotlib  # noqa: F401
        except ImportError:
            pass
    plt.rcParams['axes.unicode_minus'] = False


setup_korean_font()
print(f'프로젝트 경로: {PROJECT_DIR}')
print(f'데이터 경로: {DATA_DIR}')
print(f'기준 시점: {ANCHOR_DATE.date()}')

프로젝트 경로: c:\Users\gorhk\최종 프로젝트\finance_project\김재천\black_litterman
데이터 경로: c:\Users\gorhk\최종 프로젝트\finance_project\김재천\black_litterman\data
기준 시점: 2016-01-04


In [2]:
def resolve_gics_sector(wiki_sector: str) -> str:
    """Wikipedia GICS Sector 컬럼 값을 GICS 11 표준 명칭으로 정규화.

    Wikipedia는 이미 대부분 표준 명칭을 사용하지만, 과거 스냅샷이나
    편집 변형에 대비해 alias 테이블을 둔다.
    """
    s = str(wiki_sector).strip()
    aliases = {
        # 2018년 GICS 개편 이전 명칭
        'Telecommunication Services': 'Communication Services',
        # 표기 변이
        'Healthcare': 'Health Care',
        'Information Tech.': 'Information Technology',
    }
    return aliases.get(s, s)


def fetch_sp500_snapshot() -> pd.DataFrame:
    """Wikipedia에서 현재 S&P 500 구성 종목 테이블을 로드.

    pd.read_html(url) 직접 호출 시 Wikipedia 403 을 받을 수 있으므로
    requests 로 HTML 을 먼저 내려받은 뒤 파싱한다.

    반환 컬럼: ticker, company_name, gics_sector, gics_sub_industry
    """
    url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    resp = requests.get(url, headers=headers, timeout=30)
    resp.raise_for_status()
    tables = pd.read_html(io.StringIO(resp.text))
    raw = tables[0].copy()
    raw.columns = [c.strip() for c in raw.columns]

    # Wikipedia의 컬럼명은 시간이 지남에 따라 조금씩 변함 → 유연 매핑
    col_map = {}
    for c in raw.columns:
        cl = c.lower()
        if cl in ('symbol', 'ticker'):
            col_map[c] = 'ticker'
        elif cl in ('security', 'company'):
            col_map[c] = 'company_name'
        elif cl.startswith('gics sector'):
            col_map[c] = 'gics_sector'
        elif cl.startswith('gics sub'):
            col_map[c] = 'gics_sub_industry'

    df = raw.rename(columns=col_map)
    df = df[['ticker', 'company_name', 'gics_sector', 'gics_sub_industry']].copy()
    df['gics_sector'] = df['gics_sector'].map(resolve_gics_sector)
    # yfinance 티커 형식으로 정규화: BRK.B → BRK-B 등
    df['ticker'] = df['ticker'].str.replace('.', '-', regex=False).str.strip()
    return df.reset_index(drop=True)

In [3]:
def fetch_spy_holdings() -> tuple:
    """SPDR SPY 현재 보유 종목 xlsx 를 SSGA 공식 서버에서 다운로드.

    SPY 는 S&P 500 을 완전 복제 추적하는 ETF 로,
    운용사(State Street)가 SEC 공시 의무에 따라 매일 보유 종목 파일을 공개한다.
    따라서 이 데이터는 Wikipedia 대비 더 공식에 가까운 교차검증 소스로 사용한다.

    반환: (DataFrame[ticker, spy_sector], error_msg)
    실패 시: (None, 에러 메시지 문자열)
    """
    url = (
        'https://www.ssga.com/us/en/intermediary/etfs/library-content/'
        'products/fund-data/etfs/us/holdings-daily-us-en-spy.xlsx'
    )
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
        'Referer': 'https://www.ssga.com/',
    }
    try:
        r = requests.get(url, headers=headers, timeout=30)
        r.raise_for_status()

        # 헤더 위치 자동 탐지: 'Ticker' 또는 'Symbol' 이 있는 행을 찾는다
        df_raw = pd.read_excel(io.BytesIO(r.content), header=None)
        header_row = None
        for idx, row in df_raw.iterrows():
            vals_lower = [
                str(v).lower().strip()
                for v in row.values
                if pd.notna(v)
            ]
            if any(v in ('ticker', 'symbol') for v in vals_lower):
                header_row = idx
                break

        if header_row is None:
            return None, (
                f'헤더 행을 찾을 수 없음. '
                f'상위 6행: {df_raw.iloc[:6].values.tolist()}'
            )

        df_raw.columns = [str(c).strip() for c in df_raw.iloc[header_row]]
        df = df_raw.iloc[header_row + 1:].reset_index(drop=True).dropna(how='all')

        ticker_col = next(
            (c for c in df.columns if c.lower() in ('ticker', 'symbol')), None
        )
        sector_col = next(
            (c for c in df.columns if 'sector' in c.lower()), None
        )

        if not ticker_col or not sector_col:
            return None, f'필수 컬럼 없음. 발견 컬럼: {df.columns.tolist()}'

        result = (
            df[[ticker_col, sector_col]]
            .rename(columns={ticker_col: 'ticker', sector_col: 'spy_sector'})
            .copy()
        )
        result['ticker'] = result['ticker'].astype(str).str.strip()
        # 알파벳으로 시작하는 항목만 — 합계·각주·현금 행 제거
        result = result[
            result['ticker'].str.match(r'^[A-Z]', na=False)
        ].reset_index(drop=True)

        return result, None

    except Exception as e:
        return None, str(e)


print('fetch_spy_holdings 함수 정의 완료')


fetch_spy_holdings 함수 정의 완료


In [4]:
def estimate_historical_market_cap(ticker: str,
                                     as_of_date: pd.Timestamp = ANCHOR_DATE,
                                     max_retry: int = 2) -> Optional[dict]:
    """as_of_date 시점의 시가총액을 추정.

    추정 전략(우선순위):
      1) yfinance.Ticker.get_shares_full(): 과거 발행주식수 시계열을 직접 제공하는 종목이면 사용
      2) 분할 역산: current_shares ÷ Π(split_ratios since as_of_date)

    반환 dict 키:
      - mcap: 추정 시가총액 (USD)
      - shares: 추정 발행주식수
      - close: as_of_date 근접 영업일의 종가
      - close_date: 실제 사용된 종가 날짜 문자열
      - method: 'get_shares_full' | 'split_reversal'
      - error: 실패 사유 (정상일 경우 None)

    실패 시 None 반환.
    """
    last_err = None
    for attempt in range(max_retry + 1):
        try:
            tk = yf.Ticker(ticker)

            # (1) as_of_date 근접 영업일의 Close 확보
            hist = tk.history(
                start=(as_of_date - pd.Timedelta(days=10)).strftime('%Y-%m-%d'),
                end=(as_of_date + pd.Timedelta(days=10)).strftime('%Y-%m-%d'),
                auto_adjust=False,
                actions=False,
            )
            if hist.empty:
                return None  # 해당 시점 가격 없음 → 2016-01 미상장 가능성

            # 타임존 제거 후 as_of_date 이후 첫 영업일 선택
            hist_idx = hist.index.tz_localize(None) if hist.index.tz else hist.index
            hist = hist.copy()
            hist.index = hist_idx
            mask = hist.index >= as_of_date
            if not mask.any():
                return None
            close_idx = hist.index[mask][0]
            close_price = float(hist.loc[close_idx, 'Close'])

            # (2) shares outstanding 추정
            shares = None
            method = None

            # 전략 1: get_shares_full
            try:
                sf = tk.get_shares_full(
                    start=(as_of_date - pd.Timedelta(days=90)).strftime('%Y-%m-%d'),
                    end=(as_of_date + pd.Timedelta(days=90)).strftime('%Y-%m-%d'),
                )
                if sf is not None and len(sf) > 0:
                    sf_idx = sf.index.tz_localize(None) if sf.index.tz else sf.index
                    sf = pd.Series(sf.values, index=sf_idx)
                    # as_of_date에 가장 가까운 값
                    nearest = sf.iloc[(sf.index - as_of_date).to_series().abs().argmin()]
                    shares = float(nearest)
                    method = 'get_shares_full'
            except Exception:
                pass

            # 전략 2: 분할 역산
            if shares is None or shares <= 0:
                info = tk.info or {}
                current_shares = info.get('sharesOutstanding') or info.get('impliedSharesOutstanding')
                if not current_shares:
                    return {
                        'mcap': None, 'shares': None, 'close': close_price,
                        'close_date': close_idx.strftime('%Y-%m-%d'),
                        'method': None, 'error': 'no_shares_info',
                    }
                splits = tk.splits
                if splits is not None and len(splits) > 0:
                    sp_idx = splits.index.tz_localize(None) if splits.index.tz else splits.index
                    splits = pd.Series(splits.values, index=sp_idx)
                    # as_of_date 이후에 발생한 분할들의 누적 비율
                    post = splits[splits.index >= as_of_date]
                    cum_ratio = float(post.prod()) if len(post) > 0 else 1.0
                else:
                    cum_ratio = 1.0
                shares = float(current_shares) / cum_ratio
                method = 'split_reversal'

            return {
                'mcap': close_price * shares,
                'shares': shares,
                'close': close_price,
                'close_date': close_idx.strftime('%Y-%m-%d'),
                'method': method,
                'error': None,
            }
        except Exception as e:
            last_err = str(e)
            if attempt < max_retry:
                time.sleep(2 ** attempt)  # 1s → 2s 지수 백오프
            else:
                return {
                    'mcap': None, 'shares': None, 'close': None,
                    'close_date': None, 'method': None, 'error': last_err,
                }

In [5]:
def top_n_by_sector(universe_df: pd.DataFrame, n: int = 10) -> pd.DataFrame:
    """섹터별 시가총액 상위 N개를 선정.

    입력 DF 필수 컬럼: ticker, gics_sector, mcap_estimate
    동률 tie-break: 티커 알파벳 오름차순
    반환 DF: 상위 N개 + sector_rank 컬럼
    """
    df = universe_df.dropna(subset=['mcap_estimate']).copy()
    df = df.sort_values(
        ['gics_sector', 'mcap_estimate', 'ticker'],
        ascending=[True, False, True]
    )
    df['sector_rank'] = df.groupby('gics_sector').cumcount() + 1
    return df[df['sector_rank'] <= n].reset_index(drop=True)


print('함수 정의 완료')

함수 정의 완료


---
## Section 1. Wikipedia S&P 500 현재 구성 로드

Wikipedia의 [List of S&P 500 companies](https://en.wikipedia.org/wiki/List_of_S%26P_500_companies) 페이지를 `pd.read_html`로 파싱. GICS Sector 컬럼을 진실의 원천으로 사용한다.

In [6]:
df_wiki = fetch_sp500_snapshot()

print(f'종목 수: {len(df_wiki)}')
print('\n섹터별 종목 수:')
sector_counts = df_wiki['gics_sector'].value_counts()
print(sector_counts)

# 예상 섹터 11개와 교집합 확인
unexpected = set(df_wiki['gics_sector'].unique()) - set(GICS_11_SECTORS)
missing = set(GICS_11_SECTORS) - set(df_wiki['gics_sector'].unique())
if unexpected:
    print(f'\n경고: 비표준 섹터 발견 = {unexpected}')
if missing:
    print(f'\n경고: 누락된 표준 섹터 = {missing}')

df_wiki.head()

종목 수: 503

섹터별 종목 수:
gics_sector
Industrials               79
Financials                76
Information Technology    73
Health Care               58
Consumer Discretionary    48
Consumer Staples          36
Utilities                 31
Real Estate               31
Materials                 26
Communication Services    23
Energy                    22
Name: count, dtype: int64


,ticker,company_name,gics_sector,gics_sub_industry
0,MMM,3M,Industrials,Industrial Conglomerates
1,AOS,A. O. Smith,Industrials,Building Products
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment
3,ABBV,AbbVie,Health Care,Biotechnology
4,ACN,Accenture,Information Technology,IT Consulting & Other Services


---
## Section 1-1. SPDR SPY 교차검증

SPDR SPY 는 S&P 500 을 완전 복제 추적하는 ETF 로,
 State Street 가 **SEC 공시 의무**에 따라 매일 보유 종목 파일을 공개한다.

**용도**: 교차검증 전용 — 분석의 진실의 원천은 Wikipedia 로 유지

**확인 항목**
1. 티커 교집합율 (Wikipedia ↔ SPY)
2. GICS 섹터 일치율 — 두 소스 간 섹터 분류 차이 탐지

> SSGA 서버 정책에 따라 URL 패턴이 변경될 수 있음.
 실패 시 경고만 출력하고 Wikipedia 로 계속 진행한다.


In [7]:
spy_df, spy_err = fetch_spy_holdings()

if spy_err:
    print(f'[경고] SPY 다운로드 실패: {spy_err}')
    print('→ Wikipedia 단독으로 계속 진행합니다.')
else:
    wiki_tickers = set(df_wiki['ticker'])
    spy_tickers  = set(spy_df['ticker'])
    common    = wiki_tickers & spy_tickers
    only_wiki = sorted(wiki_tickers - spy_tickers)
    only_spy  = sorted(spy_tickers - wiki_tickers)

    print('=== 티커 커버리지 비교 ===')
    print(f'Wikipedia : {len(wiki_tickers)}개')
    print(f'SPY       : {len(spy_tickers)}개')
    print(f'교집합    : {len(common)}개  ({len(common)/max(len(wiki_tickers),1):.1%})')
    print(f'\nWikipedia에만 있는 티커 ({len(only_wiki)}개): {only_wiki}')
    print(f'SPY에만 있는 티커      ({len(only_spy)}개): {only_spy[:20]}')

    # GICS 섹터 일치율 — SPY 파일에 섹터 정보가 실제로 있는 경우에만 검증
    merged_cv = df_wiki.merge(spy_df, on='ticker', how='inner')
    spy_unique_sectors = sorted(merged_cv['spy_sector'].dropna().unique())
    spy_sectors_valid = [s for s in spy_unique_sectors if str(s).strip() not in ('', '-', 'nan')]

    if spy_sectors_valid:
        # 대소문자 차이를 허용한 소문자 비교
        merged_cv['sector_match'] = (
            merged_cv['gics_sector'].str.strip().str.lower()
            == merged_cv['spy_sector'].str.strip().str.lower()
        )
        match_rate = merged_cv['sector_match'].mean()
        n_match = int(merged_cv['sector_match'].sum())
        print(f'\n=== 섹터 일치율: {match_rate:.1%}  ({n_match} / {len(merged_cv)}) ===')

        mismatch = merged_cv.loc[
            ~merged_cv['sector_match'],
            ['ticker', 'company_name', 'gics_sector', 'spy_sector'],
        ]
        if len(mismatch) > 0:
            print(f'\n불일치 섹터 ({len(mismatch)}건) — ticker 목록:')
            print(mismatch['ticker'].tolist())
        else:
            print('모든 공통 종목의 섹터 일치')
    else:
        print(f'\n[참고] SPY 섹터 열에 유효 데이터 없음 (고유값: {spy_unique_sectors[:5]})')
        print('→ 섹터 일치율 검증 건너뜀 — 티커 커버리지만 확인')


=== 티커 커버리지 비교 ===
Wikipedia : 503개
SPY       : 503개
교집합    : 501개  (99.6%)

Wikipedia에만 있는 티커 (2개): ['BF-B', 'BRK-B']
SPY에만 있는 티커      (2개): ['BF.B', 'BRK.B']

[참고] SPY 섹터 열에 유효 데이터 없음 (고유값: ['-'])
→ 섹터 일치율 검증 건너뜀 — 티커 커버리지만 확인


---
## Section 2. 티커 리네임 처리

2016~현재 사이에 티커가 바뀐 종목들. Wikipedia는 현재 티커를 보여주므로 별도 매핑이 꼭 필요하지는 않지만, 이후 분석에서 과거 기사·데이터와 매칭할 때를 대비해 기록만 남겨둔다.

yfinance는 현재 티커로 과거 데이터를 자동 연결해주는 경우가 많지만(META는 FB 구간 포함), 일부 종목은 개별 확인 필요.

In [8]:
# (구 티커) -> (현 티커) 참고 매핑 — yfinance가 현재 티커로 이미 연결해주는 경우가 대부분
TICKER_RENAME_HISTORY = {
    'FB': 'META',     # 2022-06 사명 변경
    'VIAC': 'PARA',   # 2022-02 Paramount Global 개편
    'FISV': 'FI',     # 2023-07 Fiserv 리브랜드
    'SQ': 'XYZ',      # 2025-01 Block Inc 티커 변경
    'TWTR': None,     # 2022-10 비상장 (참고용)
}

# 현재 S&P 500에 포함되어 있는 현 티커 중에서 과거 티커를 가진 것이 있는지 점검
in_universe = df_wiki['ticker'].tolist()
for old, new in TICKER_RENAME_HISTORY.items():
    if new and new in in_universe:
        print(f'  확인: {old} → {new} (현재 S&P 500 포함)')
    elif new:
        print(f'  참고: {old} → {new} (현재 S&P 500 미포함)')
    else:
        print(f'  참고: {old} 비상장됨')

  확인: FB → META (현재 S&P 500 포함)
  참고: VIAC → PARA (현재 S&P 500 미포함)
  참고: FISV → FI (현재 S&P 500 미포함)
  확인: SQ → XYZ (현재 S&P 500 포함)
  참고: TWTR 비상장됨


---
## Section 3. 2016-01-04 시점 시가총액 추정

**경고**: 이 셀은 전체 S&P 500 (약 500 티커)에 대해 yfinance API 호출을 수행하므로 수 분~수십 분이 걸릴 수 있다. 중간 진행 상황은 50건마다 캐시에 저장되므로 중단 후 재실행해도 이어서 진행된다.

**추정 방식**
- 1순위: `get_shares_full()` 과거 발행주식수 제공 구간
- 2순위: 현재 발행주식수 ÷ 2016-01-04 이후 발생한 분할들의 누적 비율

**한계**
- 자사주 소각·신주 발행은 반영되지 않음 → 절대 시가총액은 부정확
- Top 10 **랭킹**에 대해서는 경계 종목(10~12위)을 제외하면 대체로 강건
- 2016년 당시 미상장 종목(후속 IPO)은 자동으로 배제됨(`hist.empty`)

In [9]:
mcap_cache_path = DATA_DIR / 'mcap_estimates_2016_01.pkl'

# 기존 캐시 로드
if mcap_cache_path.exists():
    with open(mcap_cache_path, 'rb') as f:
        mcap_cache = pickle.load(f)
    print(f'기존 캐시: {len(mcap_cache)}건')
else:
    mcap_cache = {}
    print('새 캐시 시작')

all_tickers = df_wiki['ticker'].tolist()
todo = [t for t in all_tickers if t not in mcap_cache]
print(f'추정 필요: {len(todo)}개 / 전체 {len(all_tickers)}개')

# 순차 호출 + 50건마다 캐시 저장 (재실행 안전성)
for i, ticker in enumerate(todo, 1):
    mcap_cache[ticker] = estimate_historical_market_cap(ticker, ANCHOR_DATE)
    if i % 25 == 0:
        print(f'  진행: {i}/{len(todo)} (마지막: {ticker})')
    if i % 50 == 0:
        with open(mcap_cache_path, 'wb') as f:
            pickle.dump(mcap_cache, f)
    # 과도한 요청 방지
    time.sleep(0.1)

# 최종 저장
with open(mcap_cache_path, 'wb') as f:
    pickle.dump(mcap_cache, f)
print(f'\n완료: 캐시 {len(mcap_cache)}건 저장 → {mcap_cache_path.name}')

기존 캐시: 33건
추정 필요: 470개 / 전체 503개


$APP: possibly delisted; no price data found  (1d 2015-12-25 -> 2016-01-14) (Yahoo error = "Data doesn't exist for startDate = 1451019600, endDate = 1452747600")


  진행: 25/470 (마지막: T)
  진행: 50/470 (마지막: BLDR)


$CARR: possibly delisted; no price data found  (1d 2015-12-25 -> 2016-01-14) (Yahoo error = "Data doesn't exist for startDate = 1451019600, endDate = 1452747600")
$CVNA: possibly delisted; no price data found  (1d 2015-12-25 -> 2016-01-14) (Yahoo error = "Data doesn't exist for startDate = 1451019600, endDate = 1452747600")


  진행: 75/470 (마지막: CHD)


...

  진행: 400/470 (마지막: TROW)


$TTD: possibly delisted; no price data found  (1d 2015-12-25 -> 2016-01-14) (Yahoo error = "Data doesn't exist for startDate = 1451019600, endDate = 1452747600")
$UBER: possibly delisted; no price data found  (1d 2015-12-25 -> 2016-01-14) (Yahoo error = "Data doesn't exist for startDate = 1451019600, endDate = 1452747600")


  진행: 425/470 (마지막: UBER)


$VLTO: possibly delisted; no price data found  (1d 2015-12-25 -> 2016-01-14) (Yahoo error = "Data doesn't exist for startDate = 1451019600, endDate = 1452747600")
$VRT: possibly delisted; no price data found  (1d 2015-12-25 -> 2016-01-14) (Yahoo error = "Data doesn't exist for startDate = 1451019600, endDate = 1452747600")
$VICI: possibly delisted; no price data found  (1d 2015-12-25 -> 2016-01-14) (Yahoo error = "Data doesn't exist for startDate = 1451019600, endDate = 1452747600")
$VST: possibly delisted; no price data found  (1d 2015-12-25 -> 2016-01-14) (Yahoo error = "Data doesn't exist for startDate = 1451019600, endDate = 1452747600")


  진행: 450/470 (마지막: WMT)

완료: 캐시 503건 저장 → mcap_estimates_2016_01.pkl


In [10]:
# 추정 결과를 DataFrame으로 집계
records = []
for ticker, res in mcap_cache.items():
    if res is None:
        records.append({
            'ticker': ticker, 'mcap_estimate': np.nan,
            'shares_estimate': np.nan, 'close_2016_01': np.nan,
            'close_date': None, 'method': None,
            'error': 'no_price_at_anchor',
        })
    else:
        records.append({
            'ticker': ticker,
            'mcap_estimate': res.get('mcap'),
            'shares_estimate': res.get('shares'),
            'close_2016_01': res.get('close'),
            'close_date': res.get('close_date'),
            'method': res.get('method'),
            'error': res.get('error'),
        })

df_mcap = pd.DataFrame(records)
df_full = df_wiki.merge(df_mcap, on='ticker', how='left')

n_ok = df_full['mcap_estimate'].notna().sum()
n_fail = (~df_full['mcap_estimate'].notna()).sum()
print(f'시가총액 추정 성공: {n_ok} / {len(df_full)}')
print(f'실패(미상장·데이터 없음 포함): {n_fail}')
print('\n추정 방식 분포:')
print(df_full['method'].value_counts(dropna=False))

df_full.head(10)

시가총액 추정 성공: 468 / 503
실패(미상장·데이터 없음 포함): 35

추정 방식 분포:
method
get_shares_full    431
split_reversal      37
None                35
Name: count, dtype: int64


,ticker,company_name,gics_sector,gics_sub_industry,mcap_estimate,shares_estimate,close_2016_01,close_date,method,error
0,MMM,3M,Industrials,Industrial Conglomerates,7.480086e+10,6.093300e+08,122.759193,2016-01-04,get_shares_full,None
1,AOS,A. O. Smith,Industrials,Building Products,3.300684e+09,8.780750e+07,37.590000,2016-01-04,get_shares_full,None
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,6.403954e+10,1.491720e+09,42.930000,2016-01-04,get_shares_full,None
3,ABBV,AbbVie,Health Care,Biotechnology,9.282354e+10,1.611240e+09,57.610001,2016-01-04,get_shares_full,None
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,2.647346e+10,2.599770e+08,101.830002,2016-01-04,get_shares_full,None
5,ADBE,Adobe Inc.,Information Technology,Application Software,4.583141e+10,4.983300e+08,91.970001,2016-01-04,get_shares_full,None
6,AMD,Advanced Micro Devices,Information Technology,Semiconductors,2.194942e+09,7.923980e+08,2.770000,2016-01-04,get_shares_full,None
7,AES,AES Corporation,Utilities,Independent Power Producers & Energy Traders,6.221282e+09,6.597330e+08,9.430000,2016-01-04,get_shares_full,None
8,AFL,Aflac,Financials,Life & Health Insurance,1.255740e+10,4.243800e+08,29.590000,2016-01-04,get_shares_full,None
9,A,Agilent Technologies,Health Care,Life Sciences Tools & Services,1.351604e+10,3.321710e+08,40.689999,2016-01-04,get_shares_full,None


---
## Section 4. 섹터별 Top 10 선정

`top_n_by_sector`로 상위 10개를 추출하고, **경계 종목(11~12위)** 도 별도로 출력해 랭킹 경계의 민감도를 눈으로 확인한다. 분할 역산 방식의 정확도 한계로 이 경계에서 순위가 바뀔 가능성이 있다.

In [11]:
# Top 10 선정
universe_top10 = top_n_by_sector(df_full, n=10)
print(f'선정 결과: {len(universe_top10)}개 (기대 110개)')

# 섹터별 개수 확인
print('\n섹터별 선정 종목 수:')
print(universe_top10.groupby('gics_sector').size().reindex(GICS_11_SECTORS))

# 섹터별 Top 10 + 경계 11~12위 비교
boundary = top_n_by_sector(df_full, n=12)
for sector in GICS_11_SECTORS:
    sub = boundary[boundary['gics_sector'] == sector].copy()
    if len(sub) == 0:
        print(f'\n[{sector}] 선정 실패')
        continue
    print(f'\n[{sector}]')
    cols = ['sector_rank', 'ticker', 'company_name', 'mcap_estimate', 'method']
    display_cols = [c for c in cols if c in sub.columns]
    print(sub[display_cols].to_string(index=False))

선정 결과: 110개 (기대 110개)

섹터별 선정 종목 수:
gics_sector
Energy                    10
Materials                 10
Industrials               10
Consumer Discretionary    10
Consumer Staples          10
Health Care               10
Financials                10
Information Technology    10
Communication Services    10
Utilities                 10
Real Estate               10
dtype: int64

[Energy]
 sector_rank ticker         company_name  mcap_estimate          method
           1    XOM           ExxonMobil   3.219238e+11 get_shares_full
           2    CVX  Chevron Corporation   1.673188e+11 get_shares_full
           3    SLB         Schlumberger   8.724290e+10 get_shares_full
           4    COP       ConocoPhillips   5.830466e+10 get_shares_full
           5    OXY Occidental Petroleum   5.140326e+10 get_shares_full
           6    BKR         Baker Hughes   4.686053e+10  split_reversal
           7    PSX          Phillips 66   4.223900e+10 get_shares_full
           8    EOG        EOG Res

---
## Section 5. `data/universe.csv` 저장

다음 노트북(`02_data_collection.ipynb`)의 입력이 되는 최종 유니버스 파일.

In [12]:
universe_out = universe_top10[[
    'ticker', 'gics_sector', 'company_name', 'sector_rank',
    'mcap_estimate', 'shares_estimate', 'close_2016_01', 'close_date', 'method'
]].copy()
universe_out = universe_out.rename(columns={'mcap_estimate': 'mcap_estimate_2016_01'})

out_path = DATA_DIR / 'universe.csv'
universe_out.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f'저장 완료: {out_path}')
print(f'행 수: {len(universe_out)}')

# 경계 종목(11~12위) 별도 저장 — 감사용
boundary_only = boundary[boundary['sector_rank'].isin([11, 12])].copy()
boundary_path = DATA_DIR / 'universe_boundary_audit.csv'
boundary_only.to_csv(boundary_path, index=False, encoding='utf-8-sig')
print(f'경계 감사 저장: {boundary_path} ({len(boundary_only)}행)')

universe_out

저장 완료: c:\Users\gorhk\최종 프로젝트\finance_project\김재천\black_litterman\data\universe.csv
행 수: 110
경계 감사 저장: c:\Users\gorhk\최종 프로젝트\finance_project\김재천\black_litterman\data\universe_boundary_audit.csv (22행)


,ticker,gics_sector,company_name,sector_rank,mcap_estimate_2016_01,shares_estimate,close_2016_01,close_date,method
0,META,Communication Services,Meta Platforms,1,2.244798e+11,2.196046e+09,102.220001,2016-01-04,split_reversal
1,VZ,Communication Services,Verizon,2,1.868670e+11,4.073840e+09,45.869999,2016-01-04,get_shares_full
2,DIS,Communication Services,Walt Disney Company (The),3,1.690561e+11,1.641640e+09,102.980003,2016-01-04,get_shares_full
3,T,Communication Services,AT&T,4,1.599454e+11,6.165000e+09,25.944109,2016-01-04,get_shares_full
4,WBD,Communication Services,Warner Bros. Discovery,5,6.620375e+10,2.506768e+09,26.410000,2016-01-04,split_reversal
...,...,...,...,...,...,...,...,...,...
105,PPL,Utilities,PPL Corporation,6,2.269550e+10,6.738570e+08,33.680000,2016-01-04,get_shares_full
106,EIX,Utilities,Edison International,7,2.269021e+10,3.849060e+08,58.950001,2016-01-04,split_reversal
107,PEG,Utilities,Public Service Enterprise Group,8,1.959903e+10,5.064350e+08,38.700001,2016-01-04,get_shares_full
108,ED,Utilities,Consolidated Edison,9,1.890713e+10,2.935890e+08,64.400002,2016-01-04,get_shares_full


---
## 요약

- `data/universe.csv`: 110 종목 (11 섹터 × 10) 확정
- `data/universe_boundary_audit.csv`: 경계(11~12위) 종목 로그
- `data/mcap_estimates_2016_01.pkl`: 추정 캐시 (재실행 시 재사용)

**다음 단계**: `02_data_collection.ipynb`에서 이 110 티커의 2016~2025 일간 가격·액션·FF5 팩터·벤치마크를 수집한다.